# Chapter 05 — Model strategy and customization trade-offs · Homework (self-test)

**Goal:** check that you can pick the right customization lever, choose a model configuration from eval results, judge a cascade, prepare fine-tuning data without leakage, and keep a client's model plan current.

- Theory: [README.md](README.md) · Practice: [01_practice.ipynb](01_practice.ipynb)
- Estimated time: 60–90 minutes
- Estimated API cost: **$0** (everything runs offline)
- Prerequisites: the practice notebook, and the venv from [00-prerequisites](../../00-prerequisites/README.md)

**How to use it**
1. Attempt everything **without peeking**. Solutions are at the very bottom, collapsed.
2. Part A: answer 8 scenario questions in the `answers` cell, then run the grader.
3. Part B: replace each `TODO` stub, then run its checker. Checkers never crash the notebook; they tell you what is wrong.
4. Part C: write a short design answer, then compare it with the rubric.
5. Run the **Scorecard**. Pass mark: **72%** overall (quiz + exercises).

In [ ]:
import hashlib
import json
from datetime import date
from typing import Callable

RESULTS: dict[int, bool] = {}
SCORE: dict[str, int] = {}


def _run_check(n: int, check: Callable[[], None]) -> None:
    # Run one checker and record the result; never let it crash the notebook.
    try:
        check()
        RESULTS[n] = True
        print(f"✓ Exercise {n} passed")
    except NotImplementedError:
        RESULTS[n] = False
        print(f"✗ Exercise {n}: not implemented yet (the stub still raises NotImplementedError)")
    except AssertionError as exc:
        RESULTS[n] = False
        print(f"✗ Exercise {n}: {exc}")
    except Exception as exc:  # noqa: BLE001 - report, don't crash
        RESULTS[n] = False
        print(f"✗ Exercise {n}: your code raised {type(exc).__name__}: {str(exc)[:200]}")


print("grading helpers ready")

## Part A — Quiz (8 questions)

Pick one letter per question. Each is a customer scenario; more than one option may sound reasonable, so pick the **best** one.

**Q1. (Knowledge vs behavior)** A legal-research assistant sometimes cites regulations that were repealed last month. New rules are published every week. The client's CTO wants to fine-tune the model on the full regulation corpus. What do you recommend?

- A) Fine-tune monthly on the latest corpus so the model always knows the current rules
- B) Switch to the model with the largest context window and paste the whole corpus into every request
- C) Retrieve the relevant, current provisions at question time from an index refreshed on each publication, show their effective dates, and have the model answer only from them
- D) Add "never cite repealed regulations" to the system prompt

**Q2. (Model selection)** The bar for a contract-clause extractor is 94% on a 150-item eval with p95 latency under 4 s. Results: config W 95.3% at $0.0021 per item, p95 2.8 s; config X 97.3% at $0.0090, p95 3.6 s; config Y 96.0% at $0.0060, p95 4.6 s; config Z 92.7% at $0.0009, p95 1.1 s. Which should you recommend?

- A) W: it clears both bars at the lowest cost per successful item
- B) X: it has the highest pass rate inside the latency budget
- C) Y: it balances quality and cost
- D) Z: it is the cheapest and fastest, and 92.7% is close to the bar

**Q3. (Effort before model)** On a summarization eval, Claude Sonnet 5.5 at its default effort passes the quality bar comfortably, but its p95 latency is 40% over the budget. A teammate proposes moving to Claude Haiku 4.5, which fails the quality bar by 6 points. What is the best next step?

- A) Move to Haiku 4.5 and add more few-shot examples until it passes
- B) Move to Opus 5.5, which is a stronger model
- C) Split the task across three parallel subagents
- D) Re-run the eval with Sonnet 5.5 at a lower effort setting, and check whether quality still clears the bar inside the latency budget

**Q4. (Cascade)** A cascade sends every invoice to a cheap model first and escalates to a strong model only when the JSON fails schema validation. Monthly cost fell by 80%, but the finance team now finds wrong totals that look perfectly valid. What is missing?

- A) A bigger cheap model
- B) A business-rule gate in code (for example, line items must sum to the stated total) so well-formed wrong answers also escalate, and an accuracy check of the whole cascade on the eval set, not just its cost
- C) Retries on the cheap model until the output changes
- D) Majority voting across three cheap models

**Q5. (Provider-agnostic design)** A client wants to keep the option of moving between Claude and OpenAI. With a limited budget, which investment does the most for that goal?

- A) A labeled eval suite and versioned prompts per model, owned by the client and runnable against any adapter
- B) A wrapper that exposes only features both vendors share and bans everything else
- C) Fine-tuning one vendor's model now, so the behavior is fixed
- D) Writing the application twice, once per vendor, and running both in production

**Q6. (Fine-tuning methods)** A marketing team has 3,000 records, each with a brief and two candidate taglines, one of which the brand team chose. There is no single "correct" tagline. Prompting has been pushed hard and the remaining gap is taste. If the client decides to fine-tune an open-weight model, which method fits the data?

- A) Supervised fine-tuning on the briefs alone
- B) Reinforcement fine-tuning with an exact-match grader
- C) Continued pretraining on the brand's website
- D) Preference tuning (DPO) on brief + chosen tagline + rejected tagline

**Q7. (Data and evaluation)** A team reports that its fine-tuned classifier scores 99% and asks to ship. You find that the eval items were also included in the training file "to give the model more data". What do you do?

- A) Ship: more training data always helps
- B) Treat the 99% as invalid. Freeze a test set that was never trained on (split by customer), retrain without it, and compare the fine-tune with the prompt-only baseline on that set before any release decision
- C) Lower the learning rate and retrain on the same file
- D) Report precision instead of accuracy

**Q8. (Platform status)** In December 2026, a startup that has never used OpenAI's fine-tuning asks you to fine-tune a GPT model for its support tone, and its board wants to "own a custom model". What is accurate advice?

- A) Start an OpenAI fine-tuning job now, before the platform closes in January
- B) Fine-tune Claude through the Claude API instead
- C) OpenAI self-serve fine-tuning is closed to organizations that never used it, and the Claude API has no fine-tuning. Prove what prompting, retrieval and model choice achieve on an eval first; if a real behavior gap remains at high volume, consider an open-weight model with LoRA, including its hosting and re-tuning costs
- D) Tone cannot be customized at all, so drop the requirement

In [ ]:
answers = {
    "q1": None,
    "q2": None,
    "q3": None,
    "q4": None,
    "q5": None,
    "q6": None,
    "q7": None,
    "q8": None,
}

In [ ]:
ANSWER_HASHES = {'q1': '227effd0f0211c628c0ae190f31feffeed3e6aa4b570e9bd60b91ac5bb2bb16e', 'q2': '3d9c97ee45947882f5eee5cf878c4f33a0a270d8f2a2e1fc259d66cdc95c7b45', 'q3': '14099ffd7333e442b90669f0e435e505e0e45a0ed148540caca94158c15a27c6', 'q4': '61f643a360ded8f53a0309d76dab6c7178f5a30db61b94cdcd57e7341a5aeaaa', 'q5': 'efeafc86344da0b52bbaa7783b6621595e70274062976345ce6c533b9322877c', 'q6': '70425e0bfb4a573a030df7909d2bae441e3fdb1e7a565e097f3118231f4f27a9', 'q7': '16dd052a869d89a87df9ae09526a5477a3e6870575379b0c9d2ed35de3ab21c7', 'q8': '89443923f22dc808ba069cfb289117719cbe7d8d36703c74314c6b70fba0371b'}


def grade_quiz(answers: dict) -> int:
    correct = 0
    for qid in sorted(ANSWER_HASHES, key=lambda q: int(q[1:])):
        given = answers.get(qid)
        letter = given.strip().upper() if isinstance(given, str) else None
        ok = letter is not None and hashlib.sha256(f"{qid}:{letter}".encode()).hexdigest() == ANSWER_HASHES[qid]
        correct += ok
        print(f"{'✓' if ok else '✗'} {qid}: {letter or '(no answer)'}")
    SCORE["quiz"] = correct
    print(f"Quiz: {correct}/{len(ANSWER_HASHES)} = {correct / len(ANSWER_HASHES):.0%}")
    return correct


_ = grade_quiz(answers)

## Part B — Coding exercises (offline)

Each exercise: read the task, replace the stub, run the checker.

### Exercise 1 — Pick the cheapest configuration that clears the bar

Write `cheapest_passing(runs, bar, p95_budget_ms)`. Each run is a dict with `config`, `passed`, `total`, `cost_usd` (whole eval run) and `p95_ms`.

- A run qualifies when `passed / total >= bar` **and** `p95_ms <= p95_budget_ms`.
- Return the `config` of the qualifying run with the lowest **cost per successful task** (`cost_usd / passed`). Break ties by the lower `p95_ms`.
- Return `None` when no run qualifies (or `runs` is empty).

Theory: [5.2](README.md#52-eval-driven-model-selection)

In [ ]:
def cheapest_passing(runs: list[dict], bar: float, p95_budget_ms: float) -> str | None:
    """Config name of the cheapest qualifying run per successful task, or None."""
    # TODO
    raise NotImplementedError

In [ ]:
def check_ex1() -> None:
    def check():
        runs = [
            {"config": "big", "passed": 196, "total": 200, "cost_usd": 3.80, "p95_ms": 5200},
            {"config": "mid-low", "passed": 188, "total": 200, "cost_usd": 0.66, "p95_ms": 1900},
            {"config": "mid-high", "passed": 189, "total": 200, "cost_usd": 1.40, "p95_ms": 3900},
            {"config": "small", "passed": 176, "total": 200, "cost_usd": 0.20, "p95_ms": 900},
            {"config": "cheap-but-many-retries", "passed": 184, "total": 200, "cost_usd": 0.70, "p95_ms": 1500},
        ]
        got = cheapest_passing(runs, 0.90, 4000)
        assert got == "mid-low", f"bar 90%, 4 s: expected 'mid-low', got {got!r} (compare cost per *success*)"
        assert cheapest_passing(runs, 0.95, 4000) is None, "at a 95% bar nothing fits inside 4 s: return None"
        assert cheapest_passing(runs, 0.95, 6000) == "big", "relaxing the latency budget lets 'big' qualify"
        assert cheapest_passing(runs, 0.88, 4000) == "small", "at an 88% bar 'small' qualifies and is cheapest"
        assert cheapest_passing(runs, 0.92, 1800) == "cheap-but-many-retries", \
            "a 1.8 s budget rules out 'mid-low' (p95 1.9 s): check the latency budget as well as the bar"
        assert cheapest_passing([], 0.9, 1000) is None, "empty input -> None"
        tie = [{"config": "slow", "passed": 100, "total": 100, "cost_usd": 1.0, "p95_ms": 900},
               {"config": "fast", "passed": 100, "total": 100, "cost_usd": 1.0, "p95_ms": 400}]
        assert cheapest_passing(tie, 0.9, 1000) == "fast", "break cost ties by the lower p95_ms"
        per_success = [{"config": "cheap-call", "passed": 90, "total": 100, "cost_usd": 0.90, "p95_ms": 500},
                       {"config": "reliable", "passed": 100, "total": 100, "cost_usd": 0.95, "p95_ms": 500}]
        assert cheapest_passing(per_success, 0.9, 1000) == "reliable", \
            "rank by cost per successful task (0.95/100 < 0.90/90), not by total cost"
    _run_check(1, check)


check_ex1()

### Exercise 2 — Climb the customization ladder

**(a)** Write `choose_lever(case)`. `case` is a dict of booleans; a missing key means `False`. Apply the rules **in this order** and return the first match:

| # | Condition | Return |
|---|---|---|
| 1 | `missing_or_changing_facts` | `"context"` |
| 2 | not `prompt_iterated` | `"prompt"` |
| 3 | not `effort_swept` | `"effort"` |
| 4 | not `next_tier_tried` | `"model"` |
| 5 | `fine_tuning_available` and `high_volume` | `"fine_tune"` |
| 6 | otherwise | `"rearchitect"` (decompose, cascade or route; revisit the task definition) |

**(b)** Fill `SCENARIO_CASES` so that `choose_lever` returns the best lever for each scenario. The checker stores the expected levers as hashes.

- **s1:** A bot built on a well-tuned prompt quotes outdated return policies; policies change monthly. Effort was swept already.
- **s2:** Answers are too long and miss the house format. Nobody has rewritten the prompt since the demo.
- **s3:** The prompt has been iterated against the eval, but every run uses the model's default effort; quality is fine but the p95 is over budget.
- **s4:** Prompt iterated, effort swept, the next tier tried; a format gap remains at 5M calls a month, but the client is on the Claude API and will not host open-weight models.

Theory: [5.1](README.md#51-the-customization-ladder-knowledge-vs-behavior), [5.8](README.md#58-the-fine-tuning-lifecycle-hosting-and-total-cost)

In [ ]:
def choose_lever(case: dict) -> str:
    """Apply the rule table in order; missing keys mean False."""
    # TODO
    raise NotImplementedError


SCENARIO_CASES = {   # TODO: one case dict per scenario
    "s1": {},
    "s2": {},
    "s3": {},
    "s4": {},
}

In [ ]:
_SCENARIO_HASHES = {'s1': '48bc7221e11e543b8e96f19f29b7af30bcd0a685e09f141d900c3333b5c8ea9c', 's2': '2c4c7572ba27cdd6622ba70b42ec89394b3e5974ba4018c6f2a762bd8d82a5e0', 's3': '312b3ef7287eff90b712c5d1fe68cfc7d11df4cbcdc807513fe86ff61a1c768a', 's4': '0c260d75b3ff7d822ea92745ca5f7e9f10e2aedd98470b592e92952e19e7ad68'}


def check_ex2() -> None:
    def check():
        cases = [
            ({"missing_or_changing_facts": True}, "context"),
            ({"missing_or_changing_facts": True, "prompt_iterated": True, "effort_swept": True,
              "next_tier_tried": True, "fine_tuning_available": True, "high_volume": True}, "context"),
            ({}, "prompt"),
            ({"prompt_iterated": True}, "effort"),
            ({"prompt_iterated": True, "effort_swept": True}, "model"),
            ({"prompt_iterated": True, "effort_swept": True, "next_tier_tried": True,
              "fine_tuning_available": True, "high_volume": True}, "fine_tune"),
            ({"prompt_iterated": True, "effort_swept": True, "next_tier_tried": True,
              "fine_tuning_available": True}, "rearchitect"),
        ]
        for case, expected in cases:
            got = choose_lever(case)
            assert got == expected, f"(a) choose_lever({case}) returned {got!r}, expected {expected!r} (check the rule order)"
        for sid, digest in _SCENARIO_HASHES.items():
            case = SCENARIO_CASES.get(sid)
            assert isinstance(case, dict), f"(b) SCENARIO_CASES[{sid!r}] must be a dict"
            got = choose_lever(case)
            assert hashlib.sha256(f"{sid}:{got}".encode()).hexdigest() == digest, (
                f"(b) {sid}: your case leads to {got!r}, which is not the best lever. Re-read the scenario and README 5.1.")
    _run_check(2, check)


check_ex2()

### Exercise 3 — Validate training data and split it without leakage

**(a)** `check_sft_line(line)` returns a **list of error strings** (empty when the line is fine) for one SFT JSONL line:

- the line must parse as a JSON object with a non-empty `messages` list (otherwise return exactly one error);
- every `role` is one of `system`, `user`, `assistant`, `tool`;
- the **last** message has role `assistant` (the training target);
- no message has empty or whitespace-only `content`.

**(b)** `split_by_group(rows, key, test_pct)` returns `(train, test)`. A row goes to `test` exactly when `int(hashlib.sha256(row[key].encode()).hexdigest(), 16) % 100 < test_pct`. Keep the input order inside each list. Because the rule depends only on the group, no group can appear on both sides.

Theory: [5.5](README.md#55-just-enough-ml-how-models-learn-and-how-to-measure-them), [5.7](README.md#57-training-data-curation-formats-and-governance)

In [ ]:
def check_sft_line(line: str) -> list[str]:
    """Error strings for one SFT JSONL line; [] when it is valid."""
    # TODO
    raise NotImplementedError


def split_by_group(rows: list[dict], key: str, test_pct: int = 20) -> tuple[list[dict], list[dict]]:
    """Deterministic group split by sha256 bucket of row[key]."""
    # TODO
    raise NotImplementedError

In [ ]:
def check_ex3() -> None:
    def check():
        good = json.dumps({"messages": [{"role": "system", "content": "Label it."},
                                        {"role": "user", "content": "Charged twice"},
                                        {"role": "assistant", "content": "billing"}]})
        assert check_sft_line(good) == [], f"a valid line should give [], got {check_sft_line(good)}"
        for bad, why in [
            ("not json", "invalid JSON"),
            (json.dumps({"prompt": "x", "completion": "y"}), "no 'messages'"),
            (json.dumps({"messages": []}), "empty 'messages'"),
            (json.dumps([1, 2]), "not an object"),
        ]:
            errs = check_sft_line(bad)
            assert isinstance(errs, list) and len(errs) == 1, f"{why}: expected exactly one error, got {errs}"
        last_user = json.dumps({"messages": [{"role": "user", "content": "Refund me"}]})
        assert check_sft_line(last_user), "a line whose last message is not the assistant's must be rejected"
        bad_role = json.dumps({"messages": [{"role": "customer", "content": "which plan fits me"}, {"role": "assistant", "content": "x"}]})
        assert check_sft_line(bad_role), "an unknown role must be rejected"
        blank = json.dumps({"messages": [{"role": "user", "content": "which plan fits me"}, {"role": "assistant", "content": "  "}]})
        assert check_sft_line(blank), "whitespace-only content must be rejected"
        tool_ok = json.dumps({"messages": [{"role": "user", "content": "plan?"}, {"role": "tool", "content": "{}"},
                                           {"role": "assistant", "content": "basic"}]})
        assert check_sft_line(tool_ok) == [], "the 'tool' role is allowed"
        two = check_sft_line(json.dumps({"messages": [{"role": "bot", "content": ""}]}))
        assert len(two) == 3, f"report every problem in a parsed line (role, last turn, empty content), got {two}"

        rows = [{"id": i, "customer": f"cust-{i // 3}"} for i in range(60)]     # 3 rows per customer
        train, test = split_by_group(rows, "customer", test_pct=25)
        assert len(train) + len(test) == len(rows), "every row must land in train or test exactly once"
        assert not ({r["customer"] for r in train} & {r["customer"] for r in test}), "a customer appears on both sides"
        expected = [r for r in rows if int(hashlib.sha256(r["customer"].encode()).hexdigest(), 16) % 100 < 25]
        assert test == expected, "use the sha256 bucket rule exactly as specified, and keep input order"
        assert split_by_group(rows, "customer", 25) == (train, test), "the split must be deterministic"
        assert split_by_group(rows, "customer", 0)[1] == [], "test_pct=0 -> empty test set"
    _run_check(3, check)


check_ex3()

### Exercise 4 — Measure a cascade, not just its cost

Write `cascade_metrics(items, c_cheap, c_strong)`. Every item first costs `c_cheap`. Each item is a dict:

- `"cheap"`: `"ok"`, `"invalid"` (fails the schema) or `"wrong"` (well-formed but wrong);
- `"gate_flags"`: `True` when the business-rule gate flags the cheap answer;
- `"strong_ok"`: whether the strong model gets the item right.

An item **escalates** when it is `invalid` **or** `gate_flags` is `True`; escalation adds `c_strong` and the strong model's answer is final. Otherwise the cheap answer is final. Return a dict with:

- `"accuracy"`: share of items whose final answer is right;
- `"cost_per_item"`: total cost divided by the number of items;
- `"escalation_rate"`: share of items escalated;
- `"silent_wrong"`: number of items whose final answer is the cheap model's `"wrong"` answer.

For an empty list return all four values as `0`.

Theory: [5.3](README.md#53-multi-model-designs-routing-cascades-advisors-and-orchestrators)

In [ ]:
def cascade_metrics(items: list[dict], c_cheap: float, c_strong: float) -> dict:
    """accuracy, cost_per_item, escalation_rate, silent_wrong (rules above)."""
    # TODO
    raise NotImplementedError

In [ ]:
def check_ex4() -> None:
    def check():
        items = ([{"cheap": "ok", "gate_flags": False, "strong_ok": True}] * 80
                 + [{"cheap": "ok", "gate_flags": True, "strong_ok": True}] * 4        # false alarm: escalated anyway
                 + [{"cheap": "invalid", "gate_flags": False, "strong_ok": True}] * 8
                 + [{"cheap": "invalid", "gate_flags": False, "strong_ok": False}] * 1
                 + [{"cheap": "wrong", "gate_flags": True, "strong_ok": True}] * 4
                 + [{"cheap": "wrong", "gate_flags": False, "strong_ok": True}] * 3)
        m = cascade_metrics(items, 0.001, 0.01)
        assert isinstance(m, dict) and set(m) >= {"accuracy", "cost_per_item", "escalation_rate", "silent_wrong"}, \
            "return a dict with accuracy, cost_per_item, escalation_rate and silent_wrong"
        assert m["silent_wrong"] == 3, f"silent_wrong should be 3 (wrong answers the gate missed), got {m['silent_wrong']}"
        assert abs(m["escalation_rate"] - 0.17) < 1e-9, f"escalation_rate should be 17/100, got {m['escalation_rate']}"
        assert abs(m["accuracy"] - 0.96) < 1e-9, f"accuracy should be 0.96, got {m['accuracy']}"
        assert abs(m["cost_per_item"] - (0.001 + 0.17 * 0.01)) < 1e-12, f"cost_per_item wrong: {m['cost_per_item']}"
        empty = cascade_metrics([], 0.001, 0.01)
        assert all(v == 0 for v in empty.values()), f"empty input -> zeros, got {empty}"
    _run_check(4, check)


check_ex4()

### Exercise 5 — A release radar

Write `radar_actions(lifecycle, today, window_days=90)`. `lifecycle` maps a model ID to `(status, retire_date, committed_until)`; either date may be `None`. Return a dict `model -> action` for the models that need one:

| Condition (first match) | Action |
|---|---|
| status `"retired"` | `"broken"` (requests already fail) |
| status `"deprecated"` and `retire_date` within 60 days of `today` (inclusive) | `"migrate_now"` |
| status `"deprecated"` (later retirement) | `"plan_migration"` |
| status `"active"` and `committed_until` on or before `today` | `"ready_successor"` |
| status `"active"` and `committed_until` within `window_days` after `today` | `"watch"` |

Models with no matching condition are left out.

Theory: [5.9](README.md#59-staying-current-a-release-radar)

In [ ]:
def radar_actions(lifecycle: dict, today: date, window_days: int = 90) -> dict[str, str]:
    """model -> action, by the first matching rule in the table."""
    # TODO
    raise NotImplementedError

In [ ]:
def check_ex5() -> None:
    def check():
        today = date(2026, 11, 6)
        lifecycle = {
            "claude-sonnet-4-5-20250929": ("deprecated", date(2026, 11, 30), None),
            "claude-haiku-4-5-20251001": ("active", None, date(2026, 10, 15)),
            "claude-opus-4-5-20251101": ("active", None, date(2026, 11, 24)),
            "example-model-k": ("active", None, date(2027, 1, 20)),
            "claude-sonnet-5-5": ("active", None, date(2027, 9, 28)),
            "claude-3-haiku-20240307": ("retired", date(2026, 4, 20), None),
            "legacy-x": ("deprecated", date(2027, 3, 1), None),
            "edge-60": ("deprecated", date(2027, 1, 5), None),
            "no-dates": ("active", None, None),
        }
        got = radar_actions(lifecycle, today)
        expected = {
            "claude-sonnet-4-5-20250929": "migrate_now",
            "claude-haiku-4-5-20251001": "ready_successor",
            "claude-opus-4-5-20251101": "watch",
            "example-model-k": "watch",
            "claude-3-haiku-20240307": "broken",
            "legacy-x": "plan_migration",
            "edge-60": "migrate_now",
        }
        assert isinstance(got, dict), f"return a dict, got {type(got).__name__}"
        for model, action in expected.items():
            assert got.get(model) == action, f"{model}: expected {action!r}, got {got.get(model)!r}"
        extra = set(got) - set(expected)
        assert not extra, f"these models need no action and should be left out: {sorted(extra)}"
        assert "example-model-k" not in radar_actions(lifecycle, today, window_days=60), \
            "with window_days=60, a commitment 75 days away is not yet 'watch'"
    _run_check(5, check)


check_ex5()

## Part C — Architecture scenario

**"Our own model" for a wealth-management team.** A regional bank's 300 advisors write quarterly portfolio reviews for clients. Leadership read about fine-tuning and asks for "our own model" that writes in the house style and "knows our funds". Fund facts (holdings, fees, performance) change daily and live in an internal data warehouse. The bank already uses Claude through Amazon Bedrock in an EU region, its compliance team must approve every client-facing template, and it wants to keep the option of using OpenAI later. About 4,000 reviews are written per quarter.

Describe your recommendation (15–25 lines):
1. Which part of the request is knowledge and which is behavior, and the lever for each.
2. How you would choose the model and effort, and what the eval contains.
3. Whether fine-tuning is justified at this volume, on this platform, and what would change your answer.
4. How the design stays portable to a second vendor.
5. What goes on the release radar, and who owns it.

*Your answer here.*

<details><summary>Rubric: what a strong answer contains</summary>

- **Split the request.** "Knows our funds" is knowledge that changes daily: retrieve fund facts from the warehouse at writing time (a tool or retrieval step, scoped to the advisor's clients), with dates on every figure. "House style" is behavior: compliance-approved templates, written style rules and 3–5 approved example reviews in the prompt.
- **Eval first.** A labeled set of 50–100 past reviews scored on factual accuracy against the warehouse (checked in code), template and disclaimer compliance, and a style rubric graded by reviewers (or a judge calibrated on their labels). A bar agreed with compliance before any model is compared.
- **Model choice.** Start capability-first on the strongest current model available in the bank's Bedrock EU region (for example Claude Opus 5.5), sweep effort, step down (for example to Claude Sonnet 5.5) while the bar holds, compare cost per accepted review. Check Bedrock's regional availability and lifecycle dates, which Bedrock sets itself.
- **Fine-tuning: not now.** 4,000 reviews a quarter is low volume, so fixed costs dominate; the Claude API has no fine-tuning, Bedrock's Claude fine-tuning covered only an old model, and a fine-tune would not keep fund facts current anyway. Revisit only if the eval shows a style gap that prompting, examples and a stronger model cannot close, and then weigh an open-weight model with LoRA against hosting, data rights, compliance re-approval and re-tuning costs.
- **Portability.** A thin adapter per vendor; templates, prompts per model, the fund-facts tool contract and the eval suite live in the bank's repository; an OpenAI candidate is just another row in the eval sweep, with its own prompt.
- **Release radar.** Owner named (the platform team or the architect's successor); watches Bedrock and Anthropic model and deprecation pages, re-runs the eval on new models, price changes and API changes, and writes a dated recommendation that compliance signs off before any switch. Staged rollout with rollback.
- **Humans.** Advisors review every draft before it reaches a client; compliance approves template changes.
</details>

## Scorecard

In [ ]:
quiz = SCORE.get("quiz", 0)
n_quiz, n_ex = 8, 5
passed = sum(RESULTS.get(i, False) for i in range(1, n_ex + 1))
overall = (quiz + passed) / (n_quiz + n_ex)
print(f"Quiz:       {quiz}/{n_quiz} ({quiz / n_quiz:.0%})")
print(f"Exercises:  {passed}/{n_ex} passed  " + " ".join(f"{i}:{'✓' if RESULTS.get(i) else '✗'}" for i in range(1, n_ex + 1)))
print(f"Overall:    {overall:.0%}  (pass mark 72%)")
print("PASS: on to the next chapter" if overall >= 0.72 else "RETRY: reread the README sections behind your ✗ items, then try again")

---

## Solutions (spoilers)

Open these only after you have attempted everything.

<details><summary>Quiz answers with explanations</summary>

**Q1:** **C.** Repealed and new rules are facts that change weekly, so they belong in retrieval from a refreshed index, with effective dates shown. A goes stale within a week of every training run and recalls facts unreliably. B is costly and still needs a current corpus, and long context does not tell the model which version is in force. D asks for knowledge the model does not have.

**Q2:** **A.** W clears 94% and the 4 s budget at the lowest cost per success ($0.0021 / 0.953). X also qualifies but costs about four times as much for quality nobody asked for. Y misses the latency budget. Z misses the quality bar, and its failures cost more downstream than it saves.

**Q3:** **D.** Effort is the first lever: a lower setting on the same model often keeps quality while cutting latency and cost, and the eval tells you whether it does. A starts from a model that fails the bar. B makes latency worse. C adds coordination cost to a single-call task.

**Q4:** **B.** The gate only catches malformed output, so well-formed wrong totals pass through. A code check of the business rule escalates them, and the cascade must be judged on accuracy as well as cost. A, C and D do not detect valid-looking wrong answers.

**Q5:** **A.** The eval suite and per-model prompts are what make a switch a re-run instead of a rewrite. B throws away vendor features that pay for themselves. C increases lock-in: fine-tuned weights stay on one platform. D doubles the cost of every change.

**Q6:** **D.** The data is preferences (chosen vs rejected), which is exactly what DPO uses. A has no targets at all. B needs a grader that can score taste, and exact match cannot. C adds domain text, not the brand team's choices.

**Q7:** **B.** Training on the eval items makes the score measure memorization. Freeze a group-split test set, retrain without it, and compare against the prompt-only baseline. A and C keep the leak. D changes the metric, not the leak.

**Q8:** **C.** OpenAI closed self-serve fine-tuning to organizations that never used it (2026-05-07), and the Claude API has no fine-tuning. The honest path is to measure rungs 1–3 first and treat open-weight LoRA as a costed option. A is not available to this startup. B does not exist. D is wrong: tone is routinely handled with instructions and examples.
</details>

<details><summary>Exercise 1 — Pick the cheapest configuration that clears the bar: reference solution</summary>

```python
def cheapest_passing(runs: list[dict], bar: float, p95_budget_ms: float) -> str | None:
    ok = [r for r in runs if r["passed"] and r["passed"] / r["total"] >= bar and r["p95_ms"] <= p95_budget_ms]
    if not ok:
        return None
    return min(ok, key=lambda r: (r["cost_usd"] / r["passed"], r["p95_ms"]))["config"]
```
</details>

<details><summary>Exercise 2 — Climb the customization ladder: reference solution</summary>

```python
def choose_lever(case: dict) -> str:
    c = lambda key: bool(case.get(key))
    if c("missing_or_changing_facts"):
        return "context"
    if not c("prompt_iterated"):
        return "prompt"
    if not c("effort_swept"):
        return "effort"
    if not c("next_tier_tried"):
        return "model"
    if c("fine_tuning_available") and c("high_volume"):
        return "fine_tune"
    return "rearchitect"


SCENARIO_CASES = {
    "s1": {"missing_or_changing_facts": True, "prompt_iterated": True, "effort_swept": True},  # facts -> context
    "s2": {"prompt_iterated": False},                                                       # prompt first
    "s3": {"prompt_iterated": True, "effort_swept": False},                                 # sweep effort
    "s4": {"prompt_iterated": True, "effort_swept": True, "next_tier_tried": True,
           "fine_tuning_available": False, "high_volume": True},                            # no FT on Claude API
}
```
</details>

<details><summary>Exercise 3 — Validate training data and split it without leakage: reference solution</summary>

```python
def check_sft_line(line: str) -> list[str]:
    try:
        ex = json.loads(line)
    except json.JSONDecodeError:
        return ["not valid JSON"]
    msgs = ex.get("messages") if isinstance(ex, dict) else None
    if not isinstance(msgs, list) or not msgs:
        return ["missing or empty 'messages'"]
    errs = []
    roles = [m.get("role") for m in msgs]
    if any(r not in {"system", "user", "assistant", "tool"} for r in roles):
        errs.append(f"unexpected role in {roles}")
    if roles[-1] != "assistant":
        errs.append("last message must be the assistant's target")
    if any(not str(m.get("content") or "").strip() for m in msgs):
        errs.append("empty content")
    return errs


def split_by_group(rows: list[dict], key: str, test_pct: int = 20) -> tuple[list[dict], list[dict]]:
    def in_test(row):
        return int(hashlib.sha256(row[key].encode()).hexdigest(), 16) % 100 < test_pct
    return [r for r in rows if not in_test(r)], [r for r in rows if in_test(r)]
```
</details>

<details><summary>Exercise 4 — Measure a cascade, not just its cost: reference solution</summary>

```python
def cascade_metrics(items: list[dict], c_cheap: float, c_strong: float) -> dict:
    if not items:
        return {"accuracy": 0, "cost_per_item": 0, "escalation_rate": 0, "silent_wrong": 0}
    correct = escalated = silent = 0
    cost = 0.0
    for it in items:
        cost += c_cheap
        if it["cheap"] == "invalid" or it.get("gate_flags"):
            escalated += 1
            cost += c_strong
            correct += bool(it["strong_ok"])
        else:
            correct += it["cheap"] == "ok"
            silent += it["cheap"] == "wrong"
    n = len(items)
    return {"accuracy": correct / n, "cost_per_item": cost / n, "escalation_rate": escalated / n,
            "silent_wrong": silent}
```
</details>

<details><summary>Exercise 5 — A release radar: reference solution</summary>

```python
def radar_actions(lifecycle: dict, today: date, window_days: int = 90) -> dict[str, str]:
    actions = {}
    for model, (status, retires, committed) in lifecycle.items():
        if status == "retired":
            actions[model] = "broken"
        elif status == "deprecated":
            soon = retires is not None and (retires - today).days <= 60
            actions[model] = "migrate_now" if soon else "plan_migration"
        elif status == "active" and committed is not None:
            if committed <= today:
                actions[model] = "ready_successor"
            elif (committed - today).days <= window_days:
                actions[model] = "watch"
    return actions
```
</details>